<a href="https://colab.research.google.com/github/shin584/project/blob/3D_simulation/NT_cas12a_model_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# [Cell 1] Nucleotide Transformer 학습을 위한 최신 라이브러리 환경 세팅
!pip install -q --upgrade transformers datasets accelerate evaluate scikit-learn scipy einops

import torch
import transformers
print("=== 환경 체크 ===")
print(f"PyTorch Version      : {torch.__version__}")
print(f"Transformers Version : {transformers.__version__} (최신 버전)")
print(f"GPU Available        : {torch.cuda.is_available()}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 116.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.9 MB/s eta 0:00:00
=== 환경 체크 ===
PyTorch Version      : 2.11.0+cu128
Transformers Version : 5.14.1 (최신 버전)
GPU Available        : True


In [2]:
# [Cell 2] Google DeepMind/InstaDeep의 표준 DNA 파운데이션 모델 토큰화
import os
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

print("=== [Phase 3.2] 데이터셋 로드 및 토큰화 (Nucleotide Transformer) ===")

required_files = ["train.csv", "val.csv", "test.csv"]
for file_name in required_files:
    if not os.path.exists(file_name):
        raise FileNotFoundError(f"'{file_name}' 파일을 찾을 수 없습니다.")

raw_datasets = load_dataset("csv", data_files={
    "train": "train.csv",
    "validation": "val.csv",
    "test": "test.csv"
})

sample_df = pd.read_csv("train.csv", nrows=1)
seq_col = next((c for c in sample_df.columns if c.lower() in ["input_sequence", "sequence", "target_sequence", "seq"]), sample_df.columns[0])
label_col = next((c for c in sample_df.columns if c.lower() in ["score", "label", "efficiency", "cleavage_efficiency", "cleavage_score"]), sample_df.columns[1])
print(f"\n매핑된 컬럼명 -> 서열: '{seq_col}' | 라벨: '{label_col}'")

# trust_remote_code=True가 전혀 필요 없는 100% Hugging Face 정식 표준 모델!
model_name = "InstaDeepAI/nucleotide-transformer-500m-human-ref"
print(f"\n'{model_name}' 토크나이저 로드 중...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    tokenized = tokenizer(
        examples[seq_col],
        padding="max_length",
        truncation=True,
        max_length=64
    )
    tokenized["label"] = [float(val) for val in examples[label_col]]
    return tokenized

print("전체 데이터셋 토큰화 적용 중...")
tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_datasets["train"].column_names
)
print("\n토큰화 완료!")

=== [Phase 3.2] 데이터셋 로드 및 토큰화 (Nucleotide Transformer) ===


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]


매핑된 컬럼명 -> 서열: 'input_sequence' | 라벨: 'score'

'InstaDeepAI/nucleotide-transformer-500m-human-ref' 토크나이저 로드 중...


...

전체 데이터셋 토큰화 적용 중...


Map:   0%|          | 0/4116 [00:00<?, ? examples/s]

Map:   0%|          | 0/514 [00:00<?, ? examples/s]

Map:   0%|          | 0/515 [00:00<?, ? examples/s]


토큰화 완료!


In [3]:
# [Cell 3] Nucleotide Transformer 회귀 헤드 안전 로딩 및 순전파 검증
import torch
from transformers import AutoModelForSequenceClassification

print("=== [Phase 3.3] 사전 학습 모델 및 회귀 헤드 로드 (표준 Native 모델) ===")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "InstaDeepAI/nucleotide-transformer-500m-human-ref"

# Hugging Face 공식 아키텍처이므로 커스텀 충돌 없이 회귀 모델(num_labels=1) 부착
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1,
    ignore_mismatched_sizes=True
)

model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n모델 로드 성공: '{model_name}' (100% Hugging Face Native Architecture)")
print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

# 샘플 텐서 순전파(Forward Pass) 검증
test_batch = {
    "input_ids": torch.tensor([tokenized_datasets["train"][0]["input_ids"]]).to(device),
    "attention_mask": torch.tensor([tokenized_datasets["train"][0]["attention_mask"]]).to(device),
    "labels": torch.tensor([tokenized_datasets["train"][0]["label"]], dtype=torch.float32).to(device)
}

model.eval()
with torch.no_grad():
    sample_output = model(**test_batch)

print("\n[샘플 텐서 순전파 테스트]")
print(f"입력 input_ids shape   : {test_batch['input_ids'].shape}")
print(f"출력 logits shape      : {sample_output.logits.shape}  --> (Batch=1, Num_Labels=1)")
print(f"초기 미학습 손실(Loss) : {sample_output.loss.item():.4f}")
print("-" * 50)
print("3단계 환경 세팅 및 샘플 검증이 완벽히 끝났습니다!")

=== [Phase 3.3] 사전 학습 모델 및 회귀 헤드 로드 (표준 Native 모델) ===


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.94GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.94GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] EsmForSequenceClassification LOAD REPORT from: InstaDeepAI/nucleotide-transformer-500m-human-ref
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.decoder.weight     | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



모델 로드 성공: 'InstaDeepAI/nucleotide-transformer-500m-human-ref' (100% Hugging Face Native Architecture)
Total Parameters     : 480,439,522
Trainable Parameters : 480,439,522

[샘플 텐서 순전파 테스트]
입력 input_ids shape   : torch.Size([1, 64])
출력 logits shape      : torch.Size([1, 1])  --> (Batch=1, Num_Labels=1)
초기 미학습 손실(Loss) : 0.9930
--------------------------------------------------
3단계 환경 세팅 및 샘플 검증이 완벽히 끝났습니다!


In [9]:
# [Cell 4] Nucleotide Transformer 파인튜닝 및 최종 평가
import numpy as np
import evaluate
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

print("=== [Phase 4] 파운데이션 모델 파인튜닝 가동 ===")

# 1. 평가 지표(Metric) 설정: 검증 시 Loss와 함께 4가지 지표 모두 산출
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.squeeze()

    if predictions.ndim > 1:
        predictions = predictions.flatten()
    if labels.ndim > 1:
        labels = labels.flatten()

    # 모델이 동일한 값만 반환할 경우의 예외 처리
    if np.std(predictions) == 0 or np.std(labels) == 0:
        return {"spearman_rho": 0.0, "pearson_r": 0.0, "mae": 0.0, "mse": 0.0}

    rho, _ = spearmanr(labels, predictions)
    r, _ = pearsonr(labels, predictions)
    mae = mean_absolute_error(labels, predictions)
    mse = mean_squared_error(labels, predictions)

    return {
        "spearman_rho": rho,
        "pearson_r": r,
        "mae": mae,
        "mse": mse
    }

# 2. 학습 하이퍼파라미터 설정
training_args = TrainingArguments(
    output_dir="./nt_finetuned_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=10,
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_dir='./logs',
    logging_steps=10
)

# 3. Trainer 객체 초기화
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# 4. 파인튜닝 학습 시작
print("\n[학습 시작] 최적의 가중치를 찾습니다...")
trainer.train()
print("\n학습 및 조기 종료(Early Stopping) 완료. 최적 가중치 로드됨.")

# 5. 최종 Test 평가 (고정된 test.csv로 1회 추론)
print("\n[최종 평가] Test 데이터셋(10%)에 대한 추론을 진행합니다.")
test_results = trainer.predict(tokenized_datasets["test"])

# Trainer가 반환하는 metrics 딕셔너리에서 값 추출 (앞에 'test_'가 붙습니다)
test_rho = test_results.metrics['test_spearman_rho']
test_pearson = test_results.metrics['test_pearson_r']
test_mae = test_results.metrics['test_mae']
test_mse = test_results.metrics['test_mse']

# 베이스라인과 동일한 포맷으로 출력
print("\n[Phase 4 파운데이션 모델 최종 성능 지표]")
print("-" * 40)
print(f"Spearman correlation (ρ) : {test_rho:.4f} (비교 핵심 지표)")
print(f"Pearson correlation (r)  : {test_pearson:.4f}")
print(f"Mean Absolute Error (MAE): {test_mae:.4f}")
print(f"Mean Squared Error (MSE) : {test_mse:.4f}")
print("-" * 40)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


=== [Phase 4] 파운데이션 모델 파인튜닝 가동 ===

[학습 시작] 최적의 가중치를 찾습니다...


Epoch,Training Loss,Validation Loss,Spearman Rho,Pearson R,Mae,Mse
1,0.014387,0.021656,0.735135,0.758338,0.115783,0.021656
2,0.013474,0.021878,0.733671,0.760802,0.116514,0.021878
3,0.012958,0.020797,0.751724,0.776193,0.111825,0.020797
4,0.010422,0.021520,0.739999,0.766436,0.112742,0.021520
5,0.010107,0.021594,0.743495,0.768225,0.116456,0.021594
6,0.010169,0.022104,0.747536,0.772149,0.114970,0.022104


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


학습 및 조기 종료(Early Stopping) 완료. 최적 가중치 로드됨.

[최종 평가] Test 데이터셋(10%)에 대한 추론을 진행합니다.



[Phase 4 파운데이션 모델 최종 성능 지표]
----------------------------------------
Spearman correlation (ρ) : 0.7867 (비교 핵심 지표)
Pearson correlation (r)  : 0.7990
Mean Absolute Error (MAE): 0.1087
Mean Squared Error (MSE) : 0.0208
----------------------------------------


In [10]:
import os
import shutil
from google.colab import drive

print("=== [최종 마무리] 모델 구글 드라이브 직접 저장 및 디스크 정리 ===")

# 1. 구글 드라이브 마운트 (이미 마운트되어 있다면 자동으로 통과됩니다)
drive.mount('/content/drive')

# 2. 구글 드라이브 내 저장 경로 설정 (원하시는 폴더명으로 수정 가능)
final_save_path = "/content/drive/MyDrive/project_shared/models/NT_cas12a_fintuned_model"

# 3. 최적 가중치 모델과 토크나이저를 구글 드라이브 경로로 직접 저장
trainer.save_model(final_save_path)
tokenizer.save_pretrained(final_save_path)
print(f"최적의 최종 모델이 구글 드라이브('{final_save_path}')에 안전하게 저장되었습니다.")

# 4. 코랩 디스크 용량(약 20GB)을 차지하는 에폭별 중간 체크포인트 통째로 삭제
checkpoint_dir = "./nt_finetuned_model"
if os.path.exists(checkpoint_dir):
    shutil.rmtree(checkpoint_dir)
    print("코랩 로컬의 불필요한 중간 체크포인트 삭제 완료! (디스크 용량 확보)")

=== [최종 마무리] 모델 구글 드라이브 직접 저장 및 디스크 정리 ===
Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

최적의 최종 모델이 구글 드라이브('/content/drive/MyDrive/project_shared/models/NT_cas12a_fintuned_model')에 안전하게 저장되었습니다.
코랩 로컬의 불필요한 중간 체크포인트 삭제 완료! (디스크 용량 확보)


In [8]:
# 1. 20GB 이상을 차지하는 에폭별 중간 체크포인트 폴더 통째로 삭제
!rm -rf ./nt_finetuned_model

# 2. (선택) 이전 학습 로그 데이터 삭제
!rm -rf ./logs

# 3. (선택) 만약 아까 저장했던 최종 모델까지 전부 지우고 0부터 다시 시작하려면 주석(#)을 지우고 실행하세요.
# !rm -rf ./best_nt_model_final

# 4. 현재 디스크 남은 용량 확인 (Use% 항목이 100%에서 줄어들었는지 확인)
!df -h /

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   51G   62G  46% /
